In [ ]:
# ==========================================
# 1. INSTALACIÓN DE LIBRERÍAS Y DESCARGA OBB
# ==========================================
!pip install -q ultralytics roboflow

from roboflow import Roboflow
from ultralytics import YOLO
from google.colab import userdata

# La clave de Roboflow va en los Secretos de Colab (nombre ROBOFLOW_API_KEY, con acceso desde el cuaderno).
# No se escribe en el cuaderno: este archivo se publica en un repositorio.
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("datascience-3acw1").project("intervertebral-disks")
version = project.version(1)
# Descarga dataset en formato OBB
dataset = version.download("yolov8-obb")


In [ ]:
# ==========================================
# 2. ENTRENAMIENTO CON ARQUITECTURA OBB
# ==========================================
# OJO: Se usa el peso pre-entrenado específico para cajas orientadas: yolov8n-obb.pt
model = YOLO("yolov8n-obb.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="disc_detector_yolo_obb",
    workers=2
)

In [ ]:
# ==========================================
# 3. GUARDAR EL MODELO EN GOOGLE DRIVE
# ==========================================
from google.colab import drive
import shutil

drive.mount('/content/drive')

# NOTA: Los resultados OBB se guardan en runs/obb/ (no en runs/detect/)
src_weights = "runs/obb/disc_detector_yolo_obb/weights/best.pt"
dst_weights = "/content/drive/MyDrive/train/best_disc_detector_obb.pt"

shutil.copy(src_weights, dst_weights)
print(f"¡Modelo OBB guardado con éxito en Drive!: {dst_weights}")

In [ ]:
# ==========================================
# 4. PRUEBA DE EXTRAER PARCHES (128x128)
# ==========================================
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Cargar pesos OBB recién entrenados
best_model = YOLO("runs/obb/disc_detector_yolo_obb/weights/best.pt")

val_images = list(Path(f"{dataset.location}/valid/images").glob("*.jpg")) + list(Path(f"{dataset.location}/valid/images").glob("*.png"))
sample_img_path = str(val_images[0])

# Inferencia
results = best_model(sample_img_path, conf=0.3)[0]

img = cv2.imread(sample_img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# En OBB las detecciones se leen desde results.obb
num_boxes = len(results.obb)
print(f"Discos OBB detectados en la muestra: {num_boxes}")

if num_boxes > 0:
    fig, axes = plt.subplots(1, num_boxes, figsize=(3 * num_boxes, 3))
    if num_boxes == 1:
        axes = [axes]

    for i, obb_item in enumerate(results.obb):
        # obb_item.xyxy devuelve las coordenadas del cuadro delimitador que envuelve la rotación
        x1, y1, x2, y2 = map(int, obb_item.xyxy[0].tolist())
        class_id = int(obb_item.cls[0])
        label = best_model.names[class_id]

        patch = img_rgb[y1:y2, x1:x2]
        if patch.size == 0:
            continue

        patch_resized = cv2.resize(patch, (128, 128), interpolation=cv2.INTER_CUBIC)

        axes[i].imshow(patch_resized)
        axes[i].set_title(f"{label}")
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# 5. VISUALIZACIÓN DE PRUEBAS DEL PACIENTE
# ==========================================
CARPETA_PRUEBA_PACIENTE = "/content/pruebas"
test_files = sorted(list(Path(CARPETA_PRUEBA_PACIENTE).glob("*.png")))

fig, axes = plt.subplots(1, len(test_files), figsize=(20, 5))

for idx, file_path in enumerate(test_files):
    results = best_model(str(file_path), conf=0.3, verbose=False)[0]

    # results.plot() dibuja automáticamente las cajas inclinadas/rotadas
    annotated_img = results.plot()

    axes[idx].imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(file_path.name, fontsize=9)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()